In [ ]:
%store -r train_df validation_df

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

SEED = 42
feature_cols = [col for col in train_df.columns if col != "Class"]

X_train = train_df[feature_cols]
X_val = validation_df[feature_cols]
y_val = validation_df["Class"]

# Fit only on training features; do not pass the Class to the model.
iso_forest = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination="auto",
    random_state=SEED,
    n_jobs=-1
)

iso_forest.fit(X_train)

"""decision_function is lower for more anomalous points, so negate it to make larger scores mean and more anomalous.""" 
val_scores = -iso_forest.decision_function(X_val)

#Find the validation threshold with the best F1. 
precision, recall, thresholds = precision_recall_curve(y_val, val_scores)

if len(thresholds) == 0:
    raise ValueError("No validation thresholds available.")


# Calculate F1 scores for each threshold and find the best one.
f1_scores = (
    2 * precision[:-1] * recall[:-1] / 
    (precision[:-1] + recall[:-1] + 1e-12)
)
best_idx = np.argmax(f1_scores)
threshold = thresholds[best_idx]

val_predictions = (val_scores >= threshold).astype(int)

print(f"Chosen validation threshold: {threshold:.6f}")
print(f"Best validation F1: {f1_scores[best_idx]:.4f}")
print("\nConfusion Matrix(rows=true, columns=predicted):")
print(confusion_matrix(y_val, val_predictions))
print("\nValidation Classification Report:")
print(classification_report(y_val, val_predictions, zero_division=0))

# Save the fitted model and validation-selected threshold for final evaluation.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "creditcard.csv").exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project root containing data/creditcard.csv.")

results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(iso_forest, results_dir / "isolation_forest.joblib")
with (results_dir / "isolation_forest_config.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "threshold": float(threshold),
            "feature_cols": feature_cols,
            "random_seed": SEED,
            "validation_f1": float(f1_scores[best_idx]),
        },
        file,
        indent=2,
    )
print(f"Saved Isolation Forest model and threshold to {results_dir}")
